# DR-NRT: Diabetic Retinopathy Detection — Kaggle Runner

This notebook runs the DR-NRT experiment pipeline on Kaggle.

**Setup:**
1. Add the APTOS dataset as a Kaggle dataset at: `meiuwu04/data-split`
2. Upload the `dr-nrt` repo as a Kaggle dataset or use Git clone
3. Set `EXP_ID` below to choose which experiment to run (0–14)

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
!pip install -q albumentations scikit-learn scipy matplotlib opencv-python-headless

In [ ]:
# ============================================================
# Cell 2: Configuration — Set Experiment ID & Paths
# ============================================================
EXP_ID = 0  # <-- Change this to run different experiments (0-14)
NUM_WORKERS = 2  # Kaggle has 4 cores

# Kaggle dataset paths
KAGGLE_DATA_ROOT = "/kaggle/input/datasets/meiuwu04/data-split/data_split"
KAGGLE_WORKING = "/kaggle/working"

In [ ]:
# ============================================================
# Cell 3: Copy source code to working directory & patch paths
# ============================================================
import shutil
import sys
from pathlib import Path

# If source is in a Kaggle dataset, copy to /kaggle/working so we can write checkpoints
src_input = Path("/kaggle/input/dr-nrt/src")
src_working = Path(KAGGLE_WORKING) / "src"

if src_input.exists() and not src_working.exists():
    shutil.copytree(src_input, src_working)
    # Also copy run_experiment.py if present
    re_input = Path("/kaggle/input/dr-nrt/run_experiment.py")
    if re_input.exists():
        shutil.copy2(re_input, Path(KAGGLE_WORKING) / "run_experiment.py")

sys.path.insert(0, KAGGLE_WORKING)

# --- Patch config paths to point at Kaggle data ---
import src.config as config_mod

kaggle_data = Path(KAGGLE_DATA_ROOT)
config_mod.DATA_DIR = kaggle_data
config_mod.TRAIN_CSV = kaggle_data / "train_label.csv"
config_mod.TEST_CSV = kaggle_data / "test_label.csv"
config_mod.TRAIN_IMG_DIR = kaggle_data / "train_split"
config_mod.TEST_IMG_DIR = kaggle_data / "test_split"
config_mod.ROOT_DIR = Path(KAGGLE_WORKING)
config_mod.CHECKPOINT_DIR = Path(KAGGLE_WORKING) / "checkpoints"
config_mod.RESULTS_DIR = Path(KAGGLE_WORKING) / "results"

# Patch dataset module (it imports at module level)
import src.dataset as dataset_mod
dataset_mod.TRAIN_CSV = config_mod.TRAIN_CSV
dataset_mod.TEST_CSV = config_mod.TEST_CSV
dataset_mod.TRAIN_IMG_DIR = config_mod.TRAIN_IMG_DIR
dataset_mod.TEST_IMG_DIR = config_mod.TEST_IMG_DIR

# Patch pseudo_label module
import src.pseudo_label as pseudo_mod
pseudo_mod.TEST_IMG_DIR = config_mod.TEST_IMG_DIR

print("Paths patched for Kaggle:")
print(f"  TRAIN_CSV:     {config_mod.TRAIN_CSV}")
print(f"  TEST_CSV:      {config_mod.TEST_CSV}")
print(f"  TRAIN_IMG_DIR: {config_mod.TRAIN_IMG_DIR}")
print(f"  TEST_IMG_DIR:  {config_mod.TEST_IMG_DIR}")
print(f"  CHECKPOINT_DIR: {config_mod.CHECKPOINT_DIR}")
print(f"  RESULTS_DIR:   {config_mod.RESULTS_DIR}")

In [ ]:
# ============================================================
# Cell 4: Import pipeline modules
# ============================================================
import logging

import torch
from torch.utils.data import DataLoader

from src.config import get_config
from src.dataset import build_datasets
from src.transforms import get_train_transform, get_val_transform
from src.train import run_training, evaluate_on_test
from src.pseudo_label import generate_pseudo_labels, finetune_with_pseudo
from src.ensemble import build_ensemble_configs, run_ensemble_inference
from src.models import build_model

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("kaggle_runner")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# Cell 5: Run Single-Model Experiment (Exp 0–13)
# ============================================================
cfg = get_config(EXP_ID)
logger.info(f"Running experiment: {cfg.exp_name}")

if EXP_ID < 14:
    transform_train = get_train_transform(cfg.aug_level)
    transform_val = get_val_transform()

    train_ds, val_ds, test_ds = build_datasets(cfg, transform_train, transform_val)
    logger.info(f"Dataset sizes — Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    model = run_training(cfg, train_loader, val_loader, device)

    if cfg.use_pseudo_labels:
        pseudo_labels = generate_pseudo_labels(model, test_ds, test_loader, cfg, device)
        model = finetune_with_pseudo(model, train_ds, pseudo_labels, cfg, device)

    metrics = evaluate_on_test(
        model, test_ds, test_loader, cfg, device,
        val_loader=val_loader if cfg.use_optimized_thresholds else None,
    )

    print(f"\n{'='*50}")
    print(f"Experiment {cfg.exp_name} — Results")
    print(f"{'='*50}")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

else:
    # Exp 14: Ensemble
    ensemble_configs = build_ensemble_configs(cfg)

    transform_train = get_train_transform(cfg.aug_level)
    transform_val = get_val_transform()

    for ecfg in ensemble_configs:
        logger.info(f"--- Training ensemble member: {ecfg.backbone} ---")
        train_ds, val_ds, test_ds = build_datasets(ecfg, transform_train, transform_val)

        train_loader = DataLoader(
            train_ds, batch_size=ecfg.batch_size, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True,
        )
        val_loader = DataLoader(
            val_ds, batch_size=ecfg.batch_size, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True,
        )
        test_loader = DataLoader(
            test_ds, batch_size=ecfg.batch_size, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True,
        )

        model = run_training(ecfg, train_loader, val_loader, device)

        if ecfg.use_pseudo_labels:
            pseudo_labels = generate_pseudo_labels(model, test_ds, test_loader, ecfg, device)
            model = finetune_with_pseudo(model, train_ds, pseudo_labels, ecfg, device)

    _, val_ds, test_ds = build_datasets(cfg, None, transform_val)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    metrics = run_ensemble_inference(
        ensemble_configs, test_ds, test_loader, device,
        val_loader=val_loader,
    )

    print(f"\n{'='*50}")
    print(f"Ensemble Experiment — Results")
    print(f"{'='*50}")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
# ============================================================
# Cell 6: Display Results — Confusion Matrix & Training Curves
# ============================================================
from IPython.display import display, Image as IPImage
from pathlib import Path

results_dir = config_mod.RESULTS_DIR / cfg.exp_name
print(f"Results saved to: {results_dir}")

# Show confusion matrix
cm_path = results_dir / "confusion_matrix.png"
if cm_path.exists():
    print("\nConfusion Matrix:")
    display(IPImage(filename=str(cm_path)))

# Show training curves
curves_path = results_dir / "training_curves.png"
if curves_path.exists():
    print("\nTraining Curves:")
    display(IPImage(filename=str(curves_path)))

# Show classification report
report_path = results_dir / "classification_report.txt"
if report_path.exists():
    print("\nClassification Report:")
    print(report_path.read_text())

# List all result files
if results_dir.exists():
    print("\nAll output files:")
    for f in sorted(results_dir.iterdir()):
        print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")